# ModelListGP

`ModelListGP` は複数の独立した GP を1つの multi-output model としてまとめるコンテナです。

出力ごとに学習入力が異なる場合、出力間 covariance を共有する必要がない場合、または出力ごとに別の GP を使いたい場合に適しています。Multi-objective Bayesian Optimization の surrogate としてもよく使われます。

## 1. Import と再現性設定

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import ModelListGP, SingleTaskGP

torch.manual_seed(0)
dtype = torch.double

## 2. 異なる学習入力を持つ2つの出力

2つの目的関数を、それぞれ異なる `X` の位置で観測したデータを作ります。

In [ ]:
def objective_1(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * x)

def objective_2(x: torch.Tensor) -> torch.Tensor:
    return 0.6 * torch.cos(2 * torch.pi * x) + 0.2 * x

train_X1 = torch.linspace(0.05, 0.95, 10, dtype=dtype).unsqueeze(-1)
train_X2 = torch.linspace(0.10, 0.90, 7, dtype=dtype).unsqueeze(-1)

train_Y1 = objective_1(train_X1) + 0.03 * torch.randn_like(train_X1)
train_Y2 = objective_2(train_X2) + 0.03 * torch.randn_like(train_X2)

## 3. 子モデルと `ModelListGP` の構築

各出力に `SingleTaskGP` を作り、それらを `ModelListGP` にまとめます。

In [ ]:
model_1 = SingleTaskGP(train_X1, train_Y1)
model_2 = SingleTaskGP(train_X2, train_Y2)
model = ModelListGP(model_1, model_2)

print("number of child models:", len(model.models))
print("supports_mll:", model.supports_mll)
print("raw_train_Xs shapes:", [x.shape for x in model.raw_train_Xs])
print("raw_train_Ys shapes:", [y.shape for y in model.raw_train_Ys])

`ModelListGP` は raw data を子モデルごとに保持します。コンテナ全体に対する単一の `raw_train_X` を新たに作ることはせず、`raw_train_Xs` / `raw_train_Ys` として公開します。

## 4. すべての子モデルをまとめて学習

`make_mll()` は `SumMarginalLogLikelihood` を返すため、BoTorch の `fit_gpytorch_mll()` で子モデルをまとめて学習できます。

In [ ]:
mll = model.make_mll()
print(type(mll).__name__)

fit_gpytorch_mll(mll)

## 5. Multi-output posterior

In [ ]:
test_X = torch.linspace(0.0, 1.0, 200, dtype=dtype).unsqueeze(-1)

model.eval()
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean
    variance = posterior.variance

print("posterior mean shape:", mean.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.scatter(train_X1.squeeze(-1), train_Y1.squeeze(-1), label="output 1 observations")
ax.scatter(train_X2.squeeze(-1), train_Y2.squeeze(-1), label="output 2 observations")
ax.plot(test_X.squeeze(-1), mean[..., 0], label="output 1 posterior")
ax.plot(test_X.squeeze(-1), mean[..., 1], label="output 2 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("ModelListGP with independent outputs")
ax.legend()
plt.show()

## 6. Multi-task GP ではなく `ModelListGP` を使う理由

`ModelListGP` の子モデルは原則として独立です。出力ごとに観測位置が異なる、出力間相関をモデル化する必要がない、出力ごとに異なる GP や transform を使いたい、multi-objective BO で目的ごとに別 surrogate を持ちたい、といった場合に適しています。

タスク間 covariance を学習して情報共有すること自体が目的なら `MultiTaskGP` や `KroneckerMultiTaskGP` を検討します。

## 7. Bayesian Optimization への接続

`ModelListGP` は multi-output Monte Carlo acquisition function に渡せます。Multi-objective BO では、qEHVI / qNEHVI などの獲得関数と objective / reference point を組み合わせる構成が代表的です。

この Notebook は surrogate model 層に焦点を当て、multi-objective acquisition の詳細例は別途扱う方針とします。